In [9]:
import pandas as pd
import numpy as np

# Step 1: Preprocessing and Weak LLM Annotations

In [2]:
df = pd.read_csv("data/speeches_with_text.csv")

In [4]:
df.columns.tolist()

['Speech_ID',
 'speaker_uid',
 'speaker_first_name',
 'speaker_last_name',
 'speaker_fullname',
 'party',
 'party_name',
 'gender',
 'source_name',
 'source_abbrev',
 'meeting_uid',
 'meeting_number',
 'meeting_datetime',
 'meeting_date',
 'session_name',
 'year',
 'legislative_period_uid',
 'Debate_ID',
 'Bill_Number',
 'Bill_Title_DE',
 'Bill_Title_FR',
 'Bill_Labels',
 'Discussion_ID',
 'Partition_ID',
 'Paragraph_ID',
 'name',
 'role',
 'language_of_text',
 'TEXT_SPAN']

In [10]:
df['leg_period'] = df['legislative_period_uid'].fillna(-1).astype(int)

In [12]:
top_parties = df['party_name'].value_counts(normalize=True).cumsum()
major_parties = top_parties[top_parties <= 0.95].index.tolist()
df['party_stratum'] = df['party_name'].apply(lambda x: x if x in major_parties else 'Other')

In [14]:
df = df.dropna(subset=['language_of_text', 'TEXT_SPAN'])
df['lang'] = df['language_of_text'].str.upper().str.strip()

In [16]:
TOTAL_SAMPLE_SIZE = 6000

In [17]:
lang_targets = {'DE': 0.50, 'FR': 0.30, 'IT': 0.20}

In [18]:
sampled_dfs = []

for lang, target_frac in lang_targets.items():
    df_lang = df[df['lang'] == lang]

    n_samples_for_lang = int(TOTAL_SAMPLE_SIZE * target_frac)

    n_samples_for_lang = min(n_samples_for_lang, len(df_lang))

    if n_samples_for_lang == 0:
        continue

    # Stratify by Party and Legislative Period WITHIN the language
    strata_cols = ['party_stratum', 'leg_period']

    # Calculate weights so each stratum is sampled proportionally
    df_lang_grouped = df_lang.groupby(strata_cols, group_keys=False)

    # Sample proportionally
    sampled_lang = df_lang_grouped.apply(
        lambda x: x.sample(
            n=int(np.round(len(x) / len(df_lang) * n_samples_for_lang)),
            random_state=42,
            replace=False
        ) if len(x) > 0 else x
    )

    sampled_dfs.append(sampled_lang)

In [19]:
df_silver_candidates = pd.concat(sampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Total sampled rows: {len(df_silver_candidates)}")
print("Language Distribution:\n", df_silver_candidates['lang'].value_counts(normalize=True))

Total sampled rows: 6001
Language Distribution:
 lang
DE    0.500250
FR    0.299617
IT    0.200133
Name: proportion, dtype: float64


In [ ]:
df = df.sort_values(by=['Discussion_ID', 'Speech_ID', 'Paragraph_ID'])

# Shift to get previous and next sentences within the SAME discussion
df['prev_span'] = df.groupby('Discussion_ID')['TEXT_SPAN'].shift(1).fillna("")
df['next_span'] = df.groupby('Discussion_ID')['TEXT_SPAN'].shift(-1).fillna("")

# Merge this context back into our sampled dataframe
df_silver_contextualized = df_silver_candidates.merge(
    df[['Discussion_ID', 'Speech_ID', 'Paragraph_ID', 'prev_span', 'next_span']],
    on=['Discussion_ID', 'Speech_ID', 'Paragraph_ID'],
    how='left'
)

# Create the exact prompt to the LLM
def format_for_llm(row):
    return (
        f"[PREVIOUS]: {row['prev_span']}\n"
        f"[TARGET]: {row['TEXT_SPAN']}\n"
        f"[NEXT]: {row['next_span']}"
    )

df_silver_contextualized['llm_prompt_text'] = df_silver_contextualized.apply(format_for_llm, axis=1)

# Inspect a specific row
print("\n--- Example LLM Input ---")
print(df_silver_contextualized['llm_prompt_text'].iloc[0])


--- Example LLM Input ---
[PREVIOUS]: Ich fasse zusammen: Folgen Sie in allen Punkten der Kommissionsmehrheit, dann haben Sie das gemacht, was sich die Kommission wünscht.
[TARGET]: Ich möchte Sie bitten, den Antrag der Minderheit Walter abzulehnen; das wäre ja der Status quo oder noch ein verstärkter Status quo. Darauf möchte ich Sie auch hinweisen: Wenn Sie das so aufnehmen würden, hätten Sie einen Widerspruch zu verschiedensten Abkommen, die Sie bereits genehmigt haben. So sieht beispielsweise das Abkommen mit Frankreich auf der Seite des Informationsinhabers keine Namen und Adressen vor, sondern das muss in irgendeiner Form identifiziert werden. Sie haben verschiedene andere Abkommen wie das mit den USA, das wiederum auf der Seite der Steuerpflichtigen sehr offen ist, wo im Grundsatz ein Name verlangt wird - "typically the name" -, was aber nicht unbedingt der Name sein muss. Wir haben eine Vielzahl von unterschiedlichen Abkommen, die entweder auf der Seite des Informationsinhaber

In [116]:
df_silver_contextualized.to_csv("silver_data_contextualized.csv", index=False)

### LLM Annotation

In [9]:
import os
import json
import pandas as pd
from tqdm import tqdm
from pydantic import BaseModel, Field
from typing import Literal
from openai import OpenAI

api_key = json.load(open("/home/matteo/.config/io.datasette.llm/keys.json"))["sdsc-key"]
client = OpenAI(api_key=api_key, base_url="https://vllm-gateway-runai-sharedllm-ralf.inference.compute.datascience.ch/v1/")

In [10]:
class DiscourseClassification(BaseModel):
    reasoning: str = Field(
        description="A brief 1-2 sentence explanation of why the target text belongs to the chosen category, considering its surrounding context."
    )
    label: Literal["FORMAL_PROCEDURE", "META_PARLIAMENTARY", "SUBSTANTIVE_ARGUMENT"] = Field(
        description="The discourse act category."
    )

In [11]:
SYSTEM_PROMPT = """
You are an expert political scientist and computational linguist analyzing transcripts from the Swiss Parliament.
Your task is to classify the [TARGET] sentence into one of three discourse acts.
You are provided with [PREVIOUS] and [NEXT] sentences to help resolve context, but you must ONLY classify the [TARGET] sentence.

TAXONOMY:
1. FORMAL_PROCEDURE: Administrative boilerplate, voting outcomes, greetings, timestamps, or speaker management.
   - Example: "Angenommen - Adopté", "[VS]", "Das Wort hat Herr Wasserfallen."

2. META_PARLIAMENTARY: Statements about the *process* of legislating, committee recommendations, or voting instructions WITHOUT providing ideological substance.
   - Example: "Die Kommission beantragt mit 13 zu 10 Stimmen, der Initiative keine Folge zu geben."
   - Example: "Ich bitte Sie, diesen Antrag abzulehnen."

3. SUBSTANTIVE_ARGUMENT: The speaker is making a claim, providing a premise, citing evidence, or expressing a political stance/ideology regarding a policy issue.
   - Example: "Diese Steuer wird den Mittelstand massiv belasten."
   - Example: "La nouvelle loi sur le CO2 est indispensable pour atteindre nos objectifs climatiques."

Be highly sensitive to the context. If the target is "Deshalb müssen wir das ablehnen." (Therefore we must reject this), look at the [PREVIOUS] text. If the previous text was a substantive policy argument, the target is the concluding claim of a SUBSTANTIVE_ARGUMENT. If the previous text was just a committee report summary, it is META_PARLIAMENTARY.
"""

In [12]:
def classify_text(text_input: str) -> dict:
    try:
        response = client.beta.chat.completions.parse(
            model="Qwen/Qwen3.6-35B-A3B-FP8",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text_input}
            ],
            response_format=DiscourseClassification,
            temperature=0.0,
            max_tokens=512,
            extra_body={
                "chat_template_kwargs": {
                    "enable_thinking": False
                }
            },
            seed=42
        )
        # Parse the structured response
        result = response.choices[0].message.parsed
        return {"reasoning": result.reasoning, "label": result.label, "error": None}

    except Exception as e:
        return {"reasoning": None, "label": None, "error": str(e)}

In [13]:
def run_annotation(input_csv: str, output_csv: str, save_interval: int = 100):
    # Load the input CSV
    df = pd.read_csv(input_csv)

    # Initialize columns if they don't exist
    if 'llm_label' not in df.columns:
        df['llm_label'] = None
        df['llm_reasoning'] = None
        df['llm_error'] = None

    # Find rows that haven't been processed yet (allows for resuming if script crashes)
    unprocessed_indices = df[df['llm_label'].isnull()].index
    print(f"Rows to process: {len(unprocessed_indices)}")

    for i, idx in enumerate(tqdm(unprocessed_indices, desc="Annotating via LLM")):
        text_input = df.at[idx, 'llm_prompt_text']

        # Call the LLM
        result = classify_text(text_input)

        # Update DataFrame
        df.at[idx, 'llm_label'] = result['label']
        df.at[idx, 'llm_reasoning'] = result['reasoning']
        df.at[idx, 'llm_error'] = result['error']

        # Save checkpoint periodically
        if (i + 1) % save_interval == 0:
            df.to_csv(output_csv, index=False)

    # Final save
    df.to_csv(output_csv, index=False)
    print(f"Annotation complete. Saved to {output_csv}.")

    # Print a quick summary
    print("\nClass Distribution:")
    print(df['llm_label'].value_counts(dropna=False))

In [14]:
run_annotation(input_csv="silver_data_contextualized.csv", output_csv="silver_data_annotated.csv", save_interval=100)

Rows to process: 4576


Annotating via LLM:   7%|▋         | 303/4576 [06:08<1:26:42,  1.22s/it]


KeyboardInterrupt: 

# Step 2: Constrastive Embeddings

In [15]:
import pandas as pd
import random
import numpy as np
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.sentence_transformer import losses
from torch.utils.data import DataLoader

In [16]:
df = pd.read_csv("silver_data_annotated.csv")
df_sub = df[df['llm_label'] == 'SUBSTANTIVE_ARGUMENT'].copy()

In [17]:
# Keep only bills that have at least 2 substantive arguments (to form pairs)
bill_counts = df_sub['Bill_Number'].value_counts()
valid_bills = bill_counts[bill_counts >= 2].index
df_sub = df_sub[df_sub['Bill_Number'].isin(valid_bills)]

In [20]:
# Construct (Anchor, Positive) Pairs and mix languages when creating pairs to force cross-lingual alignment.
train_examples = []
grouped = df_sub.groupby('Bill_Number')

for bill, group in grouped:
    sentences = group[['llm_prompt_text', 'language_of_text']].to_dict('records')

    # Create N pairs per bill (e.g., 5 pairs per bill to balance the dataset)
    n_pairs = min(5, len(sentences) // 2)

    for _ in range(n_pairs):
        s1, s2 = random.sample(sentences, 2)
        anchor = s1['llm_prompt_text']
        positive = s2['llm_prompt_text']

        print(f"\n--- Pair from Bill {bill} ---")
        print(f"Anchor (Lang: {s1['language_of_text']}): {anchor}")
        print(f"Positive (Lang: {s2['language_of_text']}): {positive}")

        train_examples.append(InputExample(texts=[anchor, positive]))

print(f"Created {len(train_examples)} contrastive pairs.")


--- Pair from Bill 2.418 ---
Anchor (Lang: DE): [PREVIOUS]: Der Kommissionssprecher hat korrekt und einlässlich die zehnjährige Geschichte der parlamentarischen Initiative Hegetschweiler dargelegt. Der Bundesrat war von Anfang an kritisch eingestellt und hat auch auf Schwächen der Vorlage hingewiesen, weil sie eben alle Infrastrukturen und nicht nur den Fluglärm erfasst. Das anvisierte Ziel der Verfahrensvereinfachung wird nicht erreicht. Im Gegenteil: Es sind sogar zusätzliche Verunsicherungen darüber entstanden, wer dann betroffen wäre; das haben die jahrelangen Arbeiten an den Vorlagen auch gezeigt.
[TARGET]: Ich bin heute eigentlich froh über den Antrag Ihrer Kommission. Mit dem Nichteintreten haben Sie nichts verloren. Die Schwächen des heutigen Systems, die vor allem das Entschädigungsverfahren betreffen, wurden erkannt. Die von Herrn Berberat zitierte interdepartementale Arbeitsgruppe kümmert sich um die Enteignungsfragen nachbarrechtlicher Ansprüche. Wir wissen hier auch, dass

In [19]:
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8, num_workers=0)

In [6]:
model = SentenceTransformer('intfloat/multilingual-e5-base')

train_loss = losses.MultipleNegativesRankingLoss(model=model)

# Fine-Tune the Encoder
print("Starting Contrastive Fine-Tuning...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=100,
    show_progress_bar=True,
    output_path="./contrastive-swiss-parliament-encoder"
)

print("Model saved to ./contrastive-swiss-parliament-encoder")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2056.54it/s]


Starting Contrastive Fine-Tuning...


/home/matteo/Documents/Projects/DemocraSci-AM/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

In [7]:
custom_model = SentenceTransformer("./contrastive-swiss-parliament-encoder")

# Embed substantive arguments
embeddings = custom_model.encode(
    df_sub['llm_prompt_text'].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

np.save("substantive_argument_embeddings.npy", embeddings)

OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './contrastive-swiss-parliament-encoder'.